# Pratique – Coleta de dados do Banco Central via API

**Fonte:** Banco Central do Brasil – SGS

**Série:** 433 – IPCA, variação mensal (%)

**Endpoint:** https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados

**Fluxo:** Requisição → DataFrame → CSV bruto → Tratamento → Validação → CSV/Parquet → Respostas 

In [1]:
# Bibliotecas usadas no projeto
from datetime import datetime   # para pegar a data de hoje
import pandas as pd             # para trabalhar com tabelas (DataFrame)
import requests                 # para fazer a chamada HTTP à API

In [2]:
# Endereço da API do Banco Central (série 433 = IPCA mensal)
URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"

# Parâmetros da consulta: formato da resposta e período desejado
PARAMS = {
    "formato": "json",
    "dataInicial": "01/01/2015",
    "dataFinal": datetime.today().strftime("%d/%m/%Y"),  # data de hoje
}

print("URL:", URL)
print("Parâmetros:", PARAMS)

URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados
Parâmetros: {'formato': 'json', 'dataInicial': '01/01/2015', 'dataFinal': '01/10/2026'}


In [3]:
# Chamada à API: timeout evita que o código trave se o servidor não responder
resposta = requests.get(URL, params=PARAMS, timeout=30)

# Se o servidor devolver erro (404, 500...), o código para aqui
resposta.raise_for_status()

print("Status HTTP:", resposta.status_code)   # 200 = sucesso
print("URL final:", resposta.url)

# Converte a resposta de JSON para uma lista de dicionários do Python
dados_json = resposta.json()
print("Registros recebidos:", len(dados_json))

# Mostra os 2 primeiros registros
dados_json[:2]

Status HTTP: 200
URL final: https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01%2F01%2F2015&dataFinal=01%2F10%2F2026
Registros recebidos: 140


[{'data': '01/01/2015', 'valor': '1.24'},
 {'data': '01/02/2015', 'valor': '1.22'}]

In [4]:
# Converte a lista de dicionários em uma tabela (DataFrame)
df_bruto = pd.DataFrame(dados_json)

# Primeiras linhas (a atividade pede essa evidência)
display(df_bruto.head())

# Tamanho da tabela: (linhas, colunas)
print("Shape:", df_bruto.shape)

# Tipos de cada coluna
print(df_bruto.dtypes)

,data,valor
0,01/01/2015,1.24
1,01/02/2015,1.22
2,01/03/2015,1.32
3,01/04/2015,0.71
4,01/05/2015,0.74


Shape: (140, 2)
data     str
valor    str
dtype: object


In [5]:
# Salva o dado bruto, exatamente como veio da API
# index=False: não grava a coluna de numeração das linhas
# utf-8-sig: garante que acentos abram certo no Excel
df_bruto.to_csv("bcb_tabela.csv", index=False, encoding="utf-8-sig")

print("Arquivo salvo: bcb_tabela.csv")

Arquivo salvo: bcb_tabela.csv


   ## Tratamentos
   1. Renomear colunas
   2. Ajustar tipos (data → datetime, valor → número)
   3. Remover valores ausentes e duplicados
   4. Criar colunas `ano` e `mes` e ordenar por data

In [6]:
# Trabalha numa cópia: o df_bruto continua intacto
df = df_bruto.copy()

# Nomes mais claros e padronizados
df = df.rename(columns={"data": "data_referencia", "valor": "ipca_mensal_pct"})

print(df.columns.tolist())

['data_referencia', 'ipca_mensal_pct']


In [7]:
# Texto "01/01/2015" vira data de verdade
df["data_referencia"] = pd.to_datetime(df["data_referencia"], format="%d/%m/%Y", errors="coerce")

# Texto "1.24" vira número decimal
# errors="coerce": o que não puder ser convertido vira NaN em vez de quebrar o código
df["ipca_mensal_pct"] = pd.to_numeric(df["ipca_mensal_pct"], errors="coerce")

print(df.dtypes)

data_referencia    datetime64[us]
ipca_mensal_pct           float64
dtype: object


In [8]:
linhas_antes = len(df)

# Remove linhas em que data ou valor estejam vazios
df = df.dropna(subset=["data_referencia", "ipca_mensal_pct"])

# Remove datas repetidas (cada mês deve aparecer uma vez)
df = df.drop_duplicates(subset=["data_referencia"])

print(f"Antes: {linhas_antes} | Depois: {len(df)} | Removidas: {linhas_antes - len(df)}")

Antes: 140 | Depois: 140 | Removidas: 0


In [9]:
# Extrai ano e mês da data para facilitar análises futuras
df["ano"] = df["data_referencia"].dt.year
df["mes"] = df["data_referencia"].dt.month

# Ordena por data e reinicia a numeração das linhas
df = df.sort_values("data_referencia").reset_index(drop=True)

display(df.head())

,data_referencia,ipca_mensal_pct,ano,mes
0,2015-01-01,1.24,2015,1
1,2015-02-01,1.22,2015,2
2,2015-03-01,1.32,2015,3
3,2015-04-01,0.71,2015,4
4,2015-05-01,0.74,2015,5


## Validação

In [10]:
print("=== Tipos de dados ===")
print(df.dtypes)

print("\n=== Valores nulos por coluna ===")
print(df.isna().sum())

print("\n=== Datas duplicadas ===")
print(df.duplicated(subset=["data_referencia"]).sum())

print("\n=== Período coberto ===")
print(df["data_referencia"].min().date(), "->", df["data_referencia"].max().date())

print("\n=== Estatísticas do IPCA mensal (%) ===")
display(df["ipca_mensal_pct"].describe())

=== Tipos de dados ===
data_referencia    datetime64[us]
ipca_mensal_pct           float64
ano                         int32
mes                         int32
dtype: object

=== Valores nulos por coluna ===
data_referencia    0
ipca_mensal_pct    0
ano                0
mes                0
dtype: int64

=== Datas duplicadas ===
0

=== Período coberto ===
2015-01-01 -> 2026-08-01

=== Estatísticas do IPCA mensal (%) ===


count    140.000000
mean       0.452786
std        0.401124
min       -0.680000
25%        0.217500
50%        0.405000
75%        0.702500
max        1.620000
Name: ipca_mensal_pct, dtype: float64

**Conclusão da validação:** os tipos foram convertidos corretamente (data para datetime e valor para float), não há valores nulos nem datas duplicadas, e a série cobre 140 meses, de jan/2015 a ago/2026. O IPCA mensal tem média de cerca de 0,45%, com mínimo negativo (deflação) e máximo de 1,62%.

In [11]:
# Salva em CSV (texto, abre no Excel)
df.to_csv("dados_tratados.csv", index=False, encoding="utf-8-sig")

# Salva em Parquet (binário e compacto, preserva os tipos das colunas)
df.to_parquet("dados_tratados.parquet", index=False)

print("Arquivos salvos: dados_tratados.csv e dados_tratados.parquet")

Arquivos salvos: dados_tratados.csv e dados_tratados.parquet


**Sobre os formatos:** o CSV é um arquivo de texto simples e não guarda o tipo das colunas (tudo vira texto até o pandas reler e interpretar). Já o **Parquet** é um formato binário e colunar, mais compacto e rápido de ler, que preserva os tipos de dados (datas continuam datas e números continuam números). Por isso, na célula abaixo, o arquivo é lido de volta para conferir que os tipos foram mantidos.

In [12]:
# Lê o Parquet de volta para provar que os tipos foram preservados
df_check = pd.read_parquet("dados_tratados.parquet")
print("Shape:", df_check.shape)
print(df_check.dtypes)

Shape: (140, 4)
data_referencia    datetime64[us]
ipca_mensal_pct           float64
ano                         int32
mes                         int32
dtype: object


## Respostas

### 1. Qual a importância do web scraping no ambiente de trabalho?

Web scraping é a coleta automatizada de dados de páginas da web. No trabalho, ele é importante porque substitui tarefas manuais de copiar e colar, economiza tempo, reduz erros e permite manter bases de dados atualizadas. Com isso, as equipes podem monitorar preços de concorrentes, acompanhar notícias, reunir indicadores e alimentar análises e dashboards com informação mais rápida para a tomada de decisão.

### 2. Riscos legais e éticos

- **Termos de uso:** muitos sites proíbem a coleta automatizada. Ignorar essas regras pode gerar bloqueio de acesso, problemas contratuais e até ações judiciais.
- **Direitos autorais:** textos, imagens e bases de dados podem ser protegidos. Coletar um conteúdo não dá o direito de republicá-lo ou redistribuí-lo.
- **LGPD:** nomes, e-mails, telefones e perfis são dados pessoais. Mesmo quando estão visíveis publicamente, sua coleta e uso exigem finalidade definida, necessidade e base legal.
- **Sobrecarga do servidor:** muitas requisições seguidas podem prejudicar o funcionamento do site, o que é um risco ético e também pode trazer consequências legais.

### 3. Duas medidas práticas de mitigação

1. **Respeitar as regras do site e limitar a coleta:** ler o robots.txt e os termos de uso antes de coletar, usar pausas entre as requisições e timeout, e dar preferência a APIs oficiais quando existirem. Isso reduz o risco de violar termos de uso e de sobrecarregar o servidor.
2. **Minimizar e proteger os dados:** coletar só o necessário, evitar dados pessoais (ou anonimizar e agregar quando forem inevitáveis), não versionar dados sensíveis (por exemplo, usando .gitignore) e documentar a fonte, a data e a finalidade da coleta. Isso reduz o risco de descumprir a LGPD.

### 4. API vs. requisição sem API

| Aspecto | Via API | Sem API (web scraping) |
|---|---|---|
| Formato | Dados estruturados (JSON/XML), documentados | HTML da página, que precisa ser interpretado (parsing) |
| Estabilidade | Estável e versionada | Quebra se o layout do site mudar |
| Permissão de uso | Uso previsto e autorizado pelo provedor | Depende dos termos de uso e do `robots.txt` |
| Manutenção | Baixa | Alta |

**Qual abordagem é mais adequada?** A API, sempre que estiver disponível. Ela é mais confiável, rápida e segura do ponto de vista legal, e entrega os dados já estruturados. Foi o caso desta atividade: o Banco Central oferece a série do IPCA por API aberta, e os dados chegaram prontos em JSON, sem precisar interpretar nenhuma página. O web scraping fica como alternativa quando não existe API nem arquivo para download, e mesmo assim exige cuidado com termos de uso, direitos autorais e LGPD.